In [ ]:
"""
Render Sentinel-2 chips into labelling PNGs.

Each output PNG shows three panels side by side:
    RGB (true colour)  |  NDVI (vegetation)  |  NDWI (water)
The two index panels carry a viridis colour bar on their right. In viridis the
high end is yellow, so the wettest (high NDWI) and greenest (high NDVI) pixels
appear yellow.

Two deliberate scaling choices, both so the SAME waterhole looks consistent from
month to month (important for temporal labelling):
  * RGB uses a FIXED reflectance stretch (DN / 10000, clipped to a reflectance
    ceiling), NOT a per-band percentile stretch. Per-band percentile stretching
    rescales each channel to its own min/max, which breaks the true R:G:B balance
    (causing magenta/lilac casts) and amplifies composite noise into speckle, and
    it makes every chip scale differently. Fixed reflectance avoids all three.
  * The index panels use TIGHT, fixed display ranges centred on where the data
    actually sits, so contrast lands on the scene (and the waterhole) instead of
    being wasted on [-1, 1] values that never occur.

One PNG is written per input .tif, using the SAME filename stem, so the chain
chip.tif -> chip.png -> chip.json (labelme) stays aligned. Training reads the
original 12-band .tif; the PNG is only a human-labelling aid.
"""

import numpy as np
import rasterio
import matplotlib
matplotlib.use("Agg")            # non-interactive backend: render straight to file, no GUI window
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# --- paths ---------------------------------------------------------------
input_tif_dir  = Path("cookie-cutting/images_tif")   # folder of 12-band Sentinel-2 GeoTIFF chips
output_png_dir = Path("cookie-cutting/images")   # where the labelling PNGs are written
output_png_dir.mkdir(exist_ok=True)

# --- band positions (1-based) from your GEE export -----------------------
# Your composite does .select(['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12']),
# so rasterio reads the bands in exactly that order:
#   1:B1  2:B2  3:B3  4:B4  5:B5  6:B6  7:B7  8:B8  9:B8A  10:B9  11:B11  12:B12
BLUE_BAND  = 2    # B2
GREEN_BAND = 3    # B3
RED_BAND   = 4    # B4
NIR_BAND   = 8    # B8  (10 m NIR)
SWIR1_BAND = 11   # B11 (only needed if you switch NDWI -> MNDWI below)

# --- RGB scaling ---------------------------------------------------------
# S2_SR_HARMONIZED stores surface reflectance x 10000, so DN / 10000 = reflectance.
REFLECTANCE_SCALE   = 1.0
# Reflectance ceiling for the stretch: visible-band land reflectance rarely exceeds
# ~0.3. Lower this (e.g. 0.25) for a brighter image, raise it if bright sand/roofs clip.
RGB_MAX_REFLECTANCE = 0.30
# Gamma < 1 brightens midtones (helps dark tropical scenes); 1.0 = linear.
RGB_GAMMA           = 0.85

# --- colour maps and display ranges --------------------------------------
NDVI_CMAP  = "YlGn"          # sequential green: greener = more vegetation
NDVI_RANGE = (-0.5, 0.8)      # positive range where vegetation lives
 
NDWI_CMAP    = "RdBu"        # diverging: dry land -> red/brown, water -> blue
NDWI_VMIN, NDWI_VCENTER, NDWI_VMAX = -0.8, 0.0, 0.6   # 0 pinned to the colour flip


def true_colour_reflectance(red, green, blue):
    """
    Build a natural-colour RGB using a FIXED reflectance stretch shared by all
    three channels, so colour balance is preserved and the result is comparable
    across chips. Returns an (H, W, 3) float array in [0, 1].
    """
    stack = np.dstack([red, green, blue]).astype("float32") / REFLECTANCE_SCALE  # -> reflectance
    stack = np.clip(stack / RGB_MAX_REFLECTANCE, 0, 1)                            # common 0..ceiling stretch
    return np.power(stack, RGB_GAMMA)                                             # gamma for midtone brightness


def normalised_difference(band_high, band_low):
    """
    Normalised difference index: (band_high - band_low) / (band_high + band_low).
    Scale-invariant, so raw DN is fine. Pixels where both bands are 0 (typically
    nodata / composite gaps) return NaN, which renders blank instead of a fake value.
    """
    band_high = band_high.astype("float32")
    band_low  = band_low.astype("float32")
    denominator = band_high + band_low
    return np.where(denominator == 0, np.nan, (band_high - band_low) / (denominator + 1e-9))


n_skipped = 0

for tif_path in sorted(input_tif_dir.glob("*.tif")):

    # --- skip chips that already have a PNG -------------------------------
    output_path = output_png_dir / f"{tif_path.stem}.png"
    if output_path.exists():
        n_skipped += 1
        continue

    # --- read only the bands we need -------------------------------------
    with rasterio.open(tif_path) as dataset:
        red   = dataset.read(RED_BAND).astype("float32")
        green = dataset.read(GREEN_BAND).astype("float32")
        blue  = dataset.read(BLUE_BAND).astype("float32")
        nir   = dataset.read(NIR_BAND).astype("float32")

    # --- true-colour composite (fixed reflectance stretch) ---------------
    true_colour = true_colour_reflectance(red, green, blue)

    # --- spectral indices ------------------------------------------------
    ndvi = normalised_difference(nir, red)     # (NIR - Red)/(NIR + Red): high = dense vegetation
    ndwi = normalised_difference(green, nir)   # (Green - NIR)/(Green + NIR): high = open water (McFeeters)
    # To use MNDWI instead (often better for water; suppresses soil/built-up), also read
    #   swir1 = dataset.read(SWIR1_BAND).astype("float32")
    # and replace the ndwi line with: ndwi = normalised_difference(green, swir1)

    # --- three-panel figure ----------------------------------------------
    figure, (rgb_axis, ndvi_axis, ndwi_axis) = plt.subplots(
        1, 3, figsize=(12, 4.5), constrained_layout=True
    )

    rgb_axis.imshow(true_colour)
    rgb_axis.set_title("RGB (true colour)")

    ndvi_image = ndvi_axis.imshow(ndvi, cmap=NDVI_CMAP, vmin=NDVI_RANGE[0], vmax=NDVI_RANGE[1])
    ndvi_axis.set_title("NDVI (vegetation)")
    figure.colorbar(ndvi_image, ax=ndvi_axis, fraction=0.046, pad=0.04, label="NDVI")
 
    # diverging norm pins 0 to the red<->blue flip regardless of the data spread
    ndwi_norm = mcolors.TwoSlopeNorm(vmin=NDWI_VMIN, vcenter=NDWI_VCENTER, vmax=NDWI_VMAX)
    ndwi_image = ndwi_axis.imshow(ndwi, cmap=NDWI_CMAP, norm=ndwi_norm)
    ndwi_axis.set_title("NDWI (water)")
    figure.colorbar(ndwi_image, ax=ndwi_axis, fraction=0.046, pad=0.04, label="NDWI")

    # drop pixel-coordinate ticks on the image panels (they only add clutter while labelling)
    for axis in (rgb_axis, ndvi_axis, ndwi_axis):
        axis.set_xticks([])
        axis.set_yticks([])

    # --- save with the SAME stem as the source .tif ----------------------
    figure.savefig(output_path, dpi=120)
    plt.close(figure)   # release the figure so memory stays flat across thousands of chips

    print(f"rendered {output_path.name}")

if n_skipped:
    print(f"skipped {n_skipped} chip(s) that already had a PNG")

rendered 2024-06_mimal_test_S2_137_S13p50_E134p58_2024-09.png
rendered 2024-06_mimal_test_S2_137_S13p50_E134p58_2024-10.png
rendered 2024-06_mimal_test_S2_137_S13p50_E134p58_2024-11.png
rendered 2024-06_mimal_test_S2_137_S13p50_E134p58_2024-12.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-01.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-02.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-03.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-04.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-05.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-06.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-07.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-08.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-09.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-10.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-11.png
rendered 2024-06_mimal_test_S2_138_S13p57_E134p54_2024-12.png
rendered